In [1]:
from autogen_ext.models.openai import AzureOpenAIChatCompletionClient

from dotenv import load_dotenv
import os

load_dotenv()

api_version = os.getenv("AZURE_OPENAI_API_VERSION")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
deployment_name = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
azure_openai_chat_completion_client = AzureOpenAIChatCompletionClient(
            model=deployment_name,
            azure_endpoint=azure_endpoint,
            api_version=api_version,
            api_key=api_key,
        )

# ⛅ Autogen: 종료 조건(termination)과 스트리밍을 갖춘 단일 에이전트 예제

이 노트북은 **`autogen_agentchat`**로 다음을 시연합니다:
- 단일 **AssistantAgent** + **툴(Function Calling)** (`get_weather`)
- **RoundRobinGroupChat** 오케스트레이션
- **TextMentionTermination("TERMINATE")** 종료 조건
- **배치 실행**(`run`)과 **스트리밍 실행**(`run_stream`)
- **팀 상태 초기화**(`reset`) 및 **Console UI** 스트리밍 출력

---

## 🧩 구성 요소

| 구성 요소 | 역할 |
|---|---|
| **AssistantAgent** | LLM 에이전트. 필요 시 등록된 툴 호출 |
| **get_weather** | 모의 날씨 조회 툴(실제 API 호출 없음) |
| **RoundRobinGroupChat** | 에이전트 팀 실행/관리 |
| **TextMentionTermination("TERMINATE")** | 답변에 특정 문자열 등장 시 대화 종료 |
| **Console** | 스트리밍 메시지를 보기 좋게 출력 |
| **run / run_stream** | 배치형 vs 스트리밍형 실행 모드 |

---

## ⚙️ 동작 흐름

1. **에이전트 생성**: `AssistantAgent(tools=[get_weather])`  
2. **종료 규칙 설정**: `TextMentionTermination("TERMINATE")`  
   - `system_message`에 *"Respond 'TERMINATE' when task is complete."* 를 명시해 모델이 종료 토큰을 출력하도록 유도  
3. **팀 구성**: `RoundRobinGroupChat([agent], termination_condition=...)`  
4. **실행**  
   - 배치형: `await team.run(task=...)`  
   - 스트리밍형:  
     - 직접 소비: `async for message in team.run_stream(...): ...`  
     - 콘솔 UI: `await Console(team.run_stream(...))`  
5. **상태 초기화**: `await team.reset()` 후 새 태스크 실행

---

In [4]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat

# Create an OpenAI model client.
model_client = azure_openai_chat_completion_client

# Define a tool that gets the weather for a city.
async def get_weather(city: str) -> str:
    """Get the weather for a city."""
    return f"The weather in {city} is 72 degrees and Sunny."


# Create an assistant agent.
weather_agent = AssistantAgent(
    "assistant",
    model_client=model_client,
    tools=[get_weather],
    system_message="Respond 'TERMINATE' when task is complete.",
)

# Define a termination condition.
text_termination = TextMentionTermination("TERMINATE")

# Create a single-agent team.
single_agent_team = RoundRobinGroupChat([weather_agent], termination_condition=text_termination)

async def run_team() -> None:
    result = await single_agent_team.run(task="What is the weather in New York?")
    print(result)


# Use `asyncio.run(run_team())` when running in a script.
await run_team()


messages=[TextMessage(id='acb9b609-9c4e-4dd0-b047-d3930b403c3c', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 10, 17, 15, 21, 38, 811978, tzinfo=datetime.timezone.utc), content='What is the weather in New York?', type='TextMessage'), ToolCallRequestEvent(id='58648b2a-4a61-4103-8616-6208e4e4247b', source='assistant', models_usage=RequestUsage(prompt_tokens=70, completion_tokens=16), metadata={}, created_at=datetime.datetime(2025, 10, 17, 15, 21, 40, 253401, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_Z4Eaz7H7Fbr17e17spzVrhuL', arguments='{"city":"New York"}', name='get_weather')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='97bb6350-d96e-4a92-b7a2-3dc2da87e3ea', source='assistant', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 10, 17, 15, 21, 40, 254332, tzinfo=datetime.timezone.utc), content=[FunctionExecutionResult(content='The weather in New York is 72 degrees and Sunny.', name='get_weather', cal

In [5]:
from autogen_agentchat.base import TaskResult


async def run_team_stream() -> None:
    async for message in single_agent_team.run_stream(task="What is the weather in New York?"):
        if isinstance(message, TaskResult):
            print("Stop Reason:", message.stop_reason)
        else:
            print(message)


# Use `asyncio.run(run_team_stream())` when running in a script.
await run_team_stream()



id='ef0b6ead-a1ca-4ef7-b78b-51ab8c8f6e3c' source='user' models_usage=None metadata={} created_at=datetime.datetime(2025, 10, 17, 15, 23, 7, 358335, tzinfo=datetime.timezone.utc) content='What is the weather in New York?' type='TextMessage'
id='080cb814-ea0c-4bbd-8a81-35d4988c88be' source='assistant' models_usage=RequestUsage(prompt_tokens=880, completion_tokens=16) metadata={} created_at=datetime.datetime(2025, 10, 17, 15, 23, 8, 571586, tzinfo=datetime.timezone.utc) content=[FunctionCall(id='call_vZE81lRJ3sMQU8Et4l1xW2Wb', arguments='{"city":"New York"}', name='get_weather')] type='ToolCallRequestEvent'
id='969def82-d6bc-433d-84d9-30bd7122599f' source='assistant' models_usage=None metadata={} created_at=datetime.datetime(2025, 10, 17, 15, 23, 8, 572917, tzinfo=datetime.timezone.utc) content=[FunctionExecutionResult(content='The weather in New York is 72 degrees and Sunny.', name='get_weather', call_id='call_vZE81lRJ3sMQU8Et4l1xW2Wb', is_error=False)] type='ToolCallExecutionEvent'
id='

In [6]:
from autogen_agentchat.ui import Console

# Use `asyncio.run(single_agent_team.reset())` when running in a script.
await single_agent_team.reset()  # Reset the team for the next run.
# Use `asyncio.run(single_agent_team.run_stream(task="What is the weather in Seattle?"))` when running in a script.
await Console(
    single_agent_team.run_stream(task="What is the weather in Seattle?")
)  # Stream the messages to the console.

---------- TextMessage (user) ----------
What is the weather in Seattle?
---------- ToolCallRequestEvent (assistant) ----------
[FunctionCall(id='call_9QSm1abGuhCRqFTTPPFP75qP', arguments='{"city":"Seattle"}', name='get_weather')]
---------- ToolCallExecutionEvent (assistant) ----------
[FunctionExecutionResult(content='The weather in Seattle is 72 degrees and Sunny.', name='get_weather', call_id='call_9QSm1abGuhCRqFTTPPFP75qP', is_error=False)]
---------- ToolCallSummaryMessage (assistant) ----------
The weather in Seattle is 72 degrees and Sunny.
---------- TextMessage (assistant) ----------
The weather in Seattle is currently 72 degrees and sunny. Is there anything else you would like to know?
---------- TextMessage (assistant) ----------
The weather in Seattle is currently 72 degrees and sunny. Would you like information about the weather in any other city?
---------- TextMessage (assistant) ----------
The weather in Seattle is currently 72 degrees and sunny. Would you like to know

TaskResult(messages=[TextMessage(id='b13f02b2-a399-484d-b2cf-ecb381e4e7c9', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 10, 17, 15, 23, 54, 175203, tzinfo=datetime.timezone.utc), content='What is the weather in Seattle?', type='TextMessage'), ToolCallRequestEvent(id='c707f76e-8619-4b6b-bd52-114509d116df', source='assistant', models_usage=RequestUsage(prompt_tokens=69, completion_tokens=15), metadata={}, created_at=datetime.datetime(2025, 10, 17, 15, 23, 55, 454884, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_9QSm1abGuhCRqFTTPPFP75qP', arguments='{"city":"Seattle"}', name='get_weather')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='000d1804-64c9-4128-ba4d-500c4cf9fbca', source='assistant', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 10, 17, 15, 23, 55, 455953, tzinfo=datetime.timezone.utc), content=[FunctionExecutionResult(content='The weather in Seattle is 72 degrees and Sunny.', name='get_weath

# ✍️ Autogen Reflection: Primary × Critic 에이전트 + 복합 종료 조건

이 노트북은 **두 에이전트(Primary/Writer ↔ Critic/Reviewer)** 가
라운드로빈으로 협업하여 결과물을 개선하고,
**승인 텍스트("APPROVE")** 또는 **최대 메시지 수**에 도달하면 종료하는
**Reflection 루프**를 시연합니다.

---

## 🧩 구성 요소

| 구성 요소 | 역할 |
|---|---|
| **AssistantAgent(primary)** | 초안/수정안 작성 |
| **AssistantAgent(critic)** | 건설적 피드백, 충족 시 `"APPROVE"` 승인 |
| **RoundRobinGroupChat** | 에이전트 순차 실행(오케스트레이션) |
| **TextMentionTermination("APPROVE")** | 승인 텍스트 등장 시 종료 |
| **MaxMessageTermination(15)** | 메시지 15개 도달 시 종료(세이프가드) |
| **Console** | 스트리밍 메시지 보기 좋게 출력 |

---

## ⚙️ 동작 흐름

1. Primary가 초안을 작성
2. Critic이 피드백 제시
3. Primary가 반영/수정
4. Critic이 `"APPROVE"`하면 종료
5. 또는 메시지가 15개면 강제 종료

---

In [10]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

# Create an OpenAI model client.
model_client = azure_openai_chat_completion_client
# Create the primary agent.
primary_agent = AssistantAgent(
    "primary",
    model_client=model_client,
    system_message="You are a helpful AI assistant.",
)

# Create the critic agent.
critic_agent = AssistantAgent(
    "critic",
    model_client=model_client,
    system_message="Provide constructive feedback. Respond with 'APPROVE' to when your feedbacks are addressed.",
)

# Define a termination condition that stops the task if the critic approves.
text_termination = TextMentionTermination("APPROVE")
# Define a termination condition that stops the task after 5 messages.
max_message_termination = MaxMessageTermination(15)
# Combine the termination conditions using the `|`` operator so that the
# task stops when either condition is met.
termination = text_termination | max_message_termination

# Create a team with the primary and critic agents.
reflection_team = RoundRobinGroupChat([primary_agent, critic_agent], termination_condition=termination)

In [11]:
# Use `asyncio.run(Console(reflection_team.run_stream(task="Write a short poem about fall season.")))` when running in a script.
await Console(
    reflection_team.run_stream(task="가을에 대한 짧은 시를 써보세요.")
)  # Stream the messages to the console.

---------- TextMessage (user) ----------
가을에 대한 짧은 시를 써보세요.


---------- TextMessage (primary) ----------
가을 바람 속에 낙엽 춤추고  
노란 빛깔 햇살 가득 내려와  
마음은 따스히 물들어가네  
조용히 다가온 가을의 노래.
---------- TextMessage (critic) ----------
아름답고 감성적인 가을 시네요. 계절의 느낌을 잘 살렸고, 낙엽과 햇살이라는 자연 요소가 시 분위기를 따뜻하게 만듭니다. 다만 몇 가지 개선할 점을 말씀드리자면:

1. '가을 바람 속에 낙엽 춤추고' 부분은 좀 더 생동감을 촉진하기 위해 '가을 바람에 낙엽은 춤추네'처럼 바꿔도 좋을 것 같습니다.

2. '노란 빛깔 햇살 가득 내려와'는 '노란 햇살이 가득히 내려와'와 같이 문맥에 조금 더 자연스럽게 어울리도록 다듬어보세요.

3. 전반적으로 음률(리듬)이 조금 불균형한 느낌이 있어서, 같은 글자 수나 운율을 맞춰 리듬감을 강화하면 더욱 아름답습니다.

종합적으로 좋은 시작이고 감성이 잘 전달되니, 위 점들을 참고하시면 더욱 완성도 높은 시가 될 것 같습니다.
---------- TextMessage (primary) ----------
소중한 피드백 감사합니다! 말씀해주신 부분을 반영하여 리듬감을 맞추고 자연스러운 표현으로 다듬은 가을 시를 다시 써보았습니다.

가을 바람에 낙엽은 춤추네  
노란 햇살이 가득히 내려와  
마음속에 찬란한 빛이 들어와  
조용히 속삭이는 가을 노래.  

더욱 감성적이고 균형 잡힌 표현이 되었길 바랍니다!
---------- TextMessage (critic) ----------
수정하신 시는 훨씬 자연스러우면서도 리듬감이 좋아졌습니다. 특히 '가을 바람에 낙엽은 춤추네'와 '노란 햇살이 가득히 내려와' 구절이 부드럽게 연결되고, '마음속에 찬란한 빛이 들어와'라는 표현이 시에 따스한 느낌을 잘 더해줍니다. 마지막 줄도 잔잔한 분위기를 잘 살려 전체적으로 감성적이고 균형 잡힌 작품이 되었습니다.

멋진 수정 작업 잘 하셨고, 가을의 분위기가 잘 전달되

TaskResult(messages=[TextMessage(id='f860570f-30b8-4d13-8801-bc029b4138e5', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 10, 17, 15, 29, 11, 286733, tzinfo=datetime.timezone.utc), content='가을에 대한 짧은 시를 써보세요.', type='TextMessage'), TextMessage(id='73844df2-304e-4db0-b268-d6c2095597d1', source='primary', models_usage=RequestUsage(prompt_tokens=33, completion_tokens=53), metadata={}, created_at=datetime.datetime(2025, 10, 17, 15, 29, 14, 168300, tzinfo=datetime.timezone.utc), content='가을 바람 속에 낙엽 춤추고  \n노란 빛깔 햇살 가득 내려와  \n마음은 따스히 물들어가네  \n조용히 다가온 가을의 노래.', type='TextMessage'), TextMessage(id='cef7593d-e73c-4ec8-adf7-3b32447f0f9a', source='critic', models_usage=RequestUsage(prompt_tokens=103, completion_tokens=240), metadata={}, created_at=datetime.datetime(2025, 10, 17, 15, 29, 21, 109821, tzinfo=datetime.timezone.utc), content="아름답고 감성적인 가을 시네요. 계절의 느낌을 잘 살렸고, 낙엽과 햇살이라는 자연 요소가 시 분위기를 따뜻하게 만듭니다. 다만 몇 가지 개선할 점을 말씀드리자면:\n\n1. '가을 바람 속에 낙엽 춤추고' 부분은 좀 더 생동

In [12]:
await Console(reflection_team.run_stream(task="중국 당나라 시풍으로 시를 써보세요."))

---------- TextMessage (user) ----------
중국 당나라 시풍으로 시를 써보세요.
---------- TextMessage (primary) ----------
산고수명(山高水明) 가을비 내리니  
단풍잎 붉게 물들어 구름 속에 흩어지네  
한 줄기 바람에 등불 흔들리고  
고요한 달빛에 마음 쉬네.
---------- TextMessage (critic) ----------
시풍을 잘 살린 아름다운 작품입니다. 당나라 시의 특징인 자연과 감정을 간결하면서도 깊이 있게 표현한 점이 돋보입니다. 다음은 몇 가지 제안을 드립니다.

1. 첫 구 '산고수명(山高水明) 가을비 내리니'는 다소 두 가지 이미지가 혼합된 느낌이 있어, 당나라 시 특유의 간결하고 함축적인 느낌을 더 살리려면 '산고수명'과 '가을비'를 좀 더 자연스럽게 연결해보는 것도 좋겠습니다.

2. '단풍잎 붉게 물들어 구름 속에 흩어지네'는 매우 아름답지만, '구름 속에 흩어지네' 구절이 약간 모호해 시상이 명확하게 전해지도록 다듬어보시면 좋겠습니다.

3. '한 줄기 바람에 등불 흔들리고'는 자연과 사람이 어우러진 묘사가 좋아서 유지해도 괜찮지만, 당나라 시는 자연과 정서의 조화를 강조하는 경향이 있으니, 등불 대신 더 자연적인 이미지를 넣어도 좋을 듯합니다.

4. 마지막 행 '고요한 달빛에 마음 쉬네'는 부드럽고 서정적인 마무리로 적절합니다.

예를 들면,

산고수명에 가을비 내리니  
단풍 붉게 물들어 저녁 구름 번지네  
한 줄기 바람에 초롱 흔들리고  
고요한 달빛에 마음 쉬네  

이렇게 다듬으면 당나라 시의 간결함과 우아함이 더 살아납니다.

전체적으로 매우 인상적이며 시풍에 맞는 작품입니다. 계속 좋은 작품 기대하겠습니다.
---------- TextMessage (primary) ----------
귀한 조언 감사합니다. 말씀해주신 방향을 반영하여 당나라 시풍에 맞게 간결하고 함축적으로 다듬은 시를 다시 쓰겠습니다.

산고수명에 가을비 내려  
단풍 붉게

TaskResult(messages=[TextMessage(id='ef6fb545-e746-4c74-90d4-d3560f57d79b', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 10, 17, 15, 29, 51, 405172, tzinfo=datetime.timezone.utc), content='중국 당나라 시풍으로 시를 써보세요.', type='TextMessage'), TextMessage(id='615cdebf-a538-4ea7-ac39-9f86b90e7ace', source='primary', models_usage=RequestUsage(prompt_tokens=640, completion_tokens=60), metadata={}, created_at=datetime.datetime(2025, 10, 17, 15, 29, 55, 143134, tzinfo=datetime.timezone.utc), content='산고수명(山高水明) 가을비 내리니  \n단풍잎 붉게 물들어 구름 속에 흩어지네  \n한 줄기 바람에 등불 흔들리고  \n고요한 달빛에 마음 쉬네.', type='TextMessage'), TextMessage(id='f96e7469-43c2-480d-8c78-b5695cf3a7db', source='critic', models_usage=RequestUsage(prompt_tokens=717, completion_tokens=397), metadata={}, created_at=datetime.datetime(2025, 10, 17, 15, 30, 11, 449085, tzinfo=datetime.timezone.utc), content="시풍을 잘 살린 아름다운 작품입니다. 당나라 시의 특징인 자연과 감정을 간결하면서도 깊이 있게 표현한 점이 돋보입니다. 다음은 몇 가지 제안을 드립니다.\n\n1. 첫 구 '산고수명(山高水明) 가을비

# 🔁 Autogen Handoff: “모르면 사용자에게 넘기기” + 복합 종료 예제

이 노트북은 **Handoff(넘겨주기) 이벤트**를 활용해  
에이전트가 **모를 때는 사용자에게 질문을 되돌리고**,  
**Handoff 발생** 또는 **"TERMINATE" 텍스트**로 종료하는  
단일 에이전트 팀 패턴을 시연합니다.

---

## 🧩 구성 요소

| 구성 요소 | 역할 |
|---|---|
| **AssistantAgent(lazy_assistant)** | 모르면 사용자에게 Handoff, 완료 시 "TERMINATE" |
| **Handoff(target="user")** | 사용자에게 대화 제어권을 넘김 |
| **HandoffTermination("user")** | 사용자에게 넘기는 순간 종료 |
| **TextMentionTermination("TERMINATE")** | 종료 텍스트 등장 시 종료 |
| **RoundRobinGroupChat** | 단일 에이전트 팀 실행 관리 |
| **Console** | 스트리밍 출력, `output_stats=True`로 통계 표시 |

---

## ⚙️ 동작 흐름

1. Lazy Assistant는 **답을 모르면** `Handoff(target="user")` 발생
2. `HandoffTermination("user")`가 트리거되어 **즉시 종료**
3. 사용자가 정보를 제공하면, 에이전트가 이를 반영하여 답변 후 **"TERMINATE"**로 종료

---

## 🧠 팁

- Handoff 대상은 **다른 에이전트(예: researcher, planner)** 로도 설정 가능  
- 종료 조건을 **OR/AND**로 조합해 안전장치(메시지 수·시간·패턴) 추가  
- 노트북 스트리밍은 UI 특성상 **일괄 출력**될 수 있음

---

In [13]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.base import Handoff
from autogen_agentchat.conditions import HandoffTermination, TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat

# Create an OpenAI model client.
model_client = azure_openai_chat_completion_client

# Create a lazy assistant agent that always hands off to the user.
lazy_agent = AssistantAgent(
    "lazy_assistant",
    model_client=model_client,
    handoffs=[Handoff(target="user", message="Transfer to user.")],
    system_message="Always transfer to user when you don't know the answer. Respond 'TERMINATE' when task is complete.",
)

# Define a termination condition that checks for handoff message targetting helper and text "TERMINATE".
handoff_termination = HandoffTermination(target="user")
text_termination = TextMentionTermination("TERMINATE")
termination = handoff_termination | text_termination

# Create a single-agent team.
lazy_agent_team = RoundRobinGroupChat([lazy_agent], termination_condition=termination)

In [ ]:
from autogen_agentchat.ui import Console

await Console(lazy_agent_team.run_stream(task="서울 날씨 어때?"),  output_stats=True)

# Handoff to the user before answering the question

await Console(lazy_agent_team.run_stream(task="날씨 맑은데? 기분이 좋다"))


---------- TextMessage (user) ----------
서울 날씨 어때?


---------- ToolCallRequestEvent (lazy_assistant) ----------
[FunctionCall(id='call_8dqtnUzfEAkzHMO4pW4gOBv6', arguments='{}', name='transfer_to_user')]
[Prompt tokens: 66, Completion tokens: 12]
---------- ToolCallExecutionEvent (lazy_assistant) ----------
[FunctionExecutionResult(content='Transfer to user.', name='transfer_to_user', call_id='call_8dqtnUzfEAkzHMO4pW4gOBv6', is_error=False)]
---------- HandoffMessage (lazy_assistant) ----------
Transfer to user.
---------- Summary ----------
Number of messages: 4
Finish reason: Handoff to user from lazy_assistant detected.
Total prompt tokens: 66
Total completion tokens: 12
Duration: 1.16 seconds
---------- TextMessage (user) ----------
날씨 맑은데? 기분이 좋다
---------- TextMessage (lazy_assistant) ----------
맑은 날씨는 정말 기분 좋게 만들죠! 좋은 하루 보내시길 바랄게요. 혹시 기분 좋게 만드는 다른 일이 있나요?
---------- TextMessage (lazy_assistant) ----------
맑은 날씨라니 정말 좋네요! 좋은 기분 계속 유지하시길 바랄게요. 오늘 특별히 계획하신 일이 있나요?
---------- TextMessage (lazy_assistant) ----------
맑은 날씨라니 기분 좋으시겠어요!